![slide53](slide/スライド53.JPG)

![slide54](slide/スライド54.JPG)

![slide55](slide/スライド55.JPG)

![slide56](slide/スライド56.JPG)

![slide57](slide/スライド57.JPG)

![slide58](slide/スライド58.JPG)

![slide59](slide/スライド59.JPG)

![slide60](slide/スライド60.JPG)

![slide61](slide/スライド61.JPG)

![slide61のgif](slide/スライド61.gif)

![slide62](slide/スライド62.JPG)

![slide63](slide/スライド63.JPG)

![slide64](slide/スライド64.JPG)

## 5-6 深層学習による手書き数字分類

![slide65](slide/スライド65.JPG)

### ライブラリのインポート

![slide66](slide/スライド66.JPG)

In [ ]:
# Pytorch関係
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms

# グラフ・画像描画用
import matplotlib.pyplot as plt

# 画像処理用
import numpy as np
import cv2

### Tensor型の利用

![slide67](slide/スライド67.JPG)

In [ ]:
x = torch.tensor([
    [1.5, 2.3, 3.8],
    [8.5, 19.4, 5.9],
    [4.8, 9.2, 8.3]
])  # Tensorの作成

print(x)
print(x.shape)  # 配列の形状の取得
print(x[1])  # 行の抜き出し
print(x[1, 2])  # 要素へのアクセス
print(x[1:3, 1:3])  # 配列のスライス (特定の行/列の切り出し)
print(x.sum())  # 要素の総和の計算

In [ ]:
x = torch.tensor(3.0, requires_grad=True)  # requires_grad=True で勾配情報を保持可能
y = 4 * x + 5.0
y.backward()  # yの式を微分
print(x.grad)  # yの式をxで微分した時の勾配

### MNISTデータの取得

![slide68](slide/スライド68.JPG)

In [ ]:
train_dataset = datasets.MNIST(
    "./mnist", train=True, download=True, transform=transforms.ToTensor())  # 学習用
test_dataset = datasets.MNIST(
    "./mnist", train=False, download=True, transform=transforms.ToTensor())  # 評価用

![slide69](slide/スライド69.JPG)

![slide70](slide/スライド70.JPG)

### ニューラルネットワークのモデルの定義

![slide71](slide/スライド71.JPG)

In [ ]:
class Net(nn.Module):  # ニューラルネットワークのモデル定義の際には、Moduleクラスを継承する

    def __init__(self, input_size, hidden_size, output_size):  # ネットワークの要素を定義
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)  # 1つ目の全結合 (入力層 -> 隠れ層) の定義
        self.fc2 = nn.Linear(hidden_size, output_size)  # 2つ目の全結合 (隠れ層 -> 出力層) の定義

    def  forward(self, x):
    # 順伝播の定義 (ネットワーク構造の定義)
    # nn.Module.forward()を上書きしており、オブジェクトのコール時に実行される
        x = self.fc1(x)
        x = torch.sigmoid(x)  # 活性化関数
        #  x = torch.relu(x)  # ReLUも試してみましょう
        x = self.fc2(x)
        return x

### 学習と評価

In [ ]:
def plot(train_losses, test_losses, accs):  # 結果プロット用関数
    fig = plt.figure(figsize=(12, 4), tight_layout=True)
    ax_loss = fig.add_subplot(1, 2, 1, xlabel="epochs", ylabel="loss")
    ax_acc = fig.add_subplot(1, 2, 2, xlabel="epochs", ylabel="accuracy")
    
    ax_loss.plot(train_losses, label="train loss")
    ax_loss.plot(test_losses, label="test loss")
    ax_acc.plot(accs, label="accuracy") 
    ax_loss.legend()
    ax_acc.legend()
    plt.show()

![slide72](slide/スライド72.JPG)

In [ ]:
def train_and_eval(n_epochs, batch_size, lr):  # エポック数, バッチサイズ, 学習率
    
    ################
    ####  設定  ####
    ################
    
    image_size = 28*28  # 画像のピクセル数
    num_classes = 10  # 分類するクラス数
    model = Net(image_size, 50, num_classes)  # ニューラルネットワークモデルの生成
    criterion = nn.CrossEntropyLoss()  # 損失関数の設定 (softmax計算も内部で行っている)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)  # 勾配計算の設定 (確率的勾配降下法)
    #  optimizer = torch.optim.Adam(model.parameters(), lr=lr)  # Adamも試してみましょう
    train_dataloader = torch.utils.data.DataLoader(  # データ読み込みの設定 (学習用)
        train_dataset,
        batch_size = batch_size,
        shuffle = True
    )
    test_dataloader = torch.utils.data.DataLoader(  # データ読み込みの設定 (評価用)
        test_dataset,
        batch_size = batch_size,
        shuffle = False
    )

    train_losses = []
    test_losses = []
    accuracies = []
    
    for epoch in range(n_epochs):  # 設定したエポック数だけ繰り返す
    
        ################
        ####  学習  ####
        ################
        
        model.train()  # モデルを訓練モードにする
        sum_train_loss = 0
        for inputs, labels in train_dataloader:  # 読み込んだデータからミニバッチを取り出す
            optimizer.zero_grad()  # 勾配計算の初期化
            inputs = inputs.view(-1, image_size)  # 画像データをベクトル化する
            labels = F.one_hot(labels, num_classes).float()  # one-hot化
            outputs = model(inputs)  # ネットワークにベクトルを入力し、出力を得る
            loss = criterion(outputs, labels)  # 損失 (出力とラベルの誤差) を計算
            sum_train_loss += loss
            loss.backward()  # 誤差逆伝播による勾配の計算
            optimizer.step()  # 重みの更新
        
        ################
        ####  評価  ####
        ################
        
        model.eval()  # モデルを評価モードにする
        sum_test_loss = 0
        correct_count = 0
        with torch.no_grad():  # 勾配計算を行わない
            for inputs, labels in test_dataloader:  # 読み込んだデータからミニバッチを取り出す
                inputs = inputs.view(-1, image_size)  # 画像データをベクトル化する
                labels = F.one_hot(labels, num_classes).float()  # one-hot化
                outputs = model(inputs)  # ネットワークにベクトルを入力し、出力を得る
                sum_test_loss += criterion(outputs, labels)  # 損失 (出力とラベルの誤差) を計算
                predicted_label = outputs.argmax(1)  # 推定ラベルを取得
                correct_count += (predicted_label.eq(labels.argmax(1).view_as(predicted_label))
                                                 .sum()
                                                 .item())  # 正解数をカウント

        train_loss = sum_train_loss.item()/len(train_dataloader)  # エポックでの平均損失の計算 (学習データ)
        test_loss = sum_test_loss.item()/len(test_dataloader)  # エポックでの平均損失の計算 (評価データ)
        accuracy = correct_count/len(test_dataset)  # エポックでの精度を計算
        status_msg = (
            f"Epoch {epoch+1}/{n_epochs} | "
            f"train_loss: {train_loss}, "
            f"test_loss: {test_loss}, "
            f"accuracy: {accuracy}")
            
        print(status_msg)
        train_losses.append(train_loss)
        test_losses.append(test_loss)
        accuracies.append(accuracy)

    torch.save(model.state_dict(), "./model.pt")  # 学習したパラメータを保存する
    
    return train_losses, test_losses, accuracies

### 学習を実行

In [ ]:
results = train_and_eval(10, 128, 1e-3)

### 学習結果の表示

In [ ]:
plot(*results)

![slide73](slide/スライド73.JPG)

### 手書き数字を分類する (評価用データで確認)

In [ ]:
check_dataloader = torch.utils.data.DataLoader(  # データ読み込みの設定 (確認用)
    test_dataset,
    batch_size = 1,
    shuffle = True
)

In [ ]:
def show_image(tensor_inputs):  # 画像を表示
    tensor_inputs = tensor_inputs.squeeze()  # サイズが1の次元を削除する
    image = tensor_inputs.detach().cpu().numpy()  # Tensorをnumpy.ndarrayにする
    image = (image*255).astype(np.uint8)  # 画像の輝度の範囲を [0.0, 1.0] から [0, 255] にする
    image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)  # 白黒画像をカラー画像のフォーマットにして、画像を表示できるようにする
    image = cv2.resize(image, (64, 64))  # 画像をすこし大きくリサイズ
    
    plt.figure(figsize=(1, 1))
    plt.axis("off")
    plt.imshow(image)
    plt.show()

![slide74](slide/スライド74.JPG)

In [ ]:
def check_io(n_show=5):  # 入力画像と、それに対する出力を確認
    image_size = 28*28  # 画像のピクセル数
    model = Net(image_size, 50, 10)  # ニューラルネットワークモデルの生成
    model.load_state_dict(torch.load("./model.pt"))  # 学習したパラメータを読み込む
    model.eval()  # モデルを評価モードにする
    with torch.no_grad():  # 勾配計算を行わない
        for i, (inputs, _) in enumerate(check_dataloader):  # 読み込んだデータからミニバッチを取り出す
            if i >= n_show:
                return
            show_image(inputs)  # 入力画像を表示
            inputs = inputs.view(-1, image_size)  # 画像データをベクトル化する
            outputs = model(inputs)  # ネットワークにベクトルを入力し、出力を得る
            outputs = F.softmax(outputs, dim=1)  #  ソフトマックスで各ラベルに属する確率を出力させる
            print(f"I think the above image is {outputs.argmax(1).item()}!")  # 結果の表示

In [ ]:
check_io()  # 実行するたびに結果が変わります

### 手書き数字を分類する (オリジナルデータで確認)

![slide75](slide/スライド75.JPG)

In [ ]:
def load_original_data():  # オリジナルデータの読み込み
    inputs = cv2.imread("./your_number.png")  # オリジナルデータを読み込む
    inputs = cv2.cvtColor(inputs, cv2.COLOR_BGR2GRAY)  # 読み込んだ画像を白黒画像に変更する
    inputs = torch.tensor(inputs).to(torch.float32)  # numpy.ndarrayをtorch.Tensorに変換する
    inputs = inputs / 255  # 画像の輝度の範囲を [0, 255] から [0.0, 1.0] にする
    inputs = inputs.unsqueeze(0).unsqueeze(0)  # モデルの入力サイズにそろえる [25, 25] -> [1, 1, 25, 25]

    return inputs

In [ ]:
def check_io_with_original_data():  # 入力画像と、それに対する出力を確認
    image_size = 28*28  # 画像のピクセル数
    model = Net(image_size, 50, 10)  # ニューラルネットワークモデルの生成
    model.load_state_dict(torch.load("./model.pt"))  # 学習したパラメータを読み込む
    model.eval()  # モデルを評価モードにする
    with torch.no_grad():  # 勾配計算を行わない
        inputs = load_original_data()  # オリジナルデータを読み込む
        show_image(inputs)  # 入力画像を表示
        inputs = inputs.view(-1, image_size)  # 画像データをベクトル化する
        outputs = model(inputs)  # ネットワークにベクトルを入力し、出力を得る
        outputs = F.softmax(outputs, dim=1)  #  ソフトマックスで各ラベルに属する確率を出力させる
        print(f"I think the above image is {outputs.argmax(1).item()}!")  # 結果の表示

In [ ]:
check_io_with_original_data()

![slide76](slide/スライド76.JPG)

![slide77](slide/スライド77.JPG)